# Evaluate Graph Visualizer on STAFF III

Whole-recording NeuroKit delineation + knowledge-graph rules, scored against **balloon-up intervals** and **occluded artery**.

STAFF does not label STEMI millivolts. A true positive is a detection **during balloon inflation**; a territory hit is a detection in leads (or a diagnosis) that match `occluded_artery`.

The 10 s cap in the Graph Viewer is a **UI** limit. NeuroKit `ecg_delineate(method="dwt")` has no documented maximum length. This notebook delineates **the entire file** via `symptom_detection.evaluate_recording`. Set `WINDOW_S` only if a run runs out of memory.

Helpers: `evaluation/staff_eval.py`. Detector code: Graph Visualizer `symptom_detection/evaluate_recording.py`.

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd() if (Path.cwd() / "evaluation" / "common.py").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / "evaluation"))

THESIS = REPO.parent.parent
BACKEND = THESIS / "Graph_Visualizer" / "graph-viewer" / "backend"
sys.path.insert(0, str(BACKEND))

import pandas as pd
from IPython.display import display

import common as C
import staff_eval as E

from symptom_detection.AnomalyDetector import AnomalyDetector
from symptom_detection.evaluate_recording import evaluate_recording, signal_from_arrays, slice_signal
from symptom_detection.ecg_features import compute_ecg_metrics
from symptom_detection.delineation_runtime import get_active_strategy_name

# --- config (edit these) ---
GRAPH_PATH = (
    THESIS / "LangChain" / "data" / "generated" / "runs" / "run_20260726_131455" / "Graph.graphml"
)
POSITIVE = E.PositiveSpec(
    diagnosis_names=None,
    rule_names=None,
    diagnosis_patterns=["STEMI", "occlusion"],
    rule_patterns=[],
)
WINDOW_S = None
WINDOW_START_S = 0.0
CACHE = True
OVERWRITE_CACHE = False
PROC_DIR = C.PROCESSED_ROOT / "staff_iii"
CACHE_DIR = PROC_DIR / "eval_cache"

print("graph:", GRAPH_PATH, "exists=", GRAPH_PATH.is_file())
print("strategy:", get_active_strategy_name())
print("processed:", PROC_DIR)

## Graph inventory — tick the positive-label set

In [ ]:
detector = AnomalyDetector.from_graphml(GRAPH_PATH)
inv = E.inventory_graph(detector.graph)
print("node counts:\n", inv["type"].value_counts().to_string())
positives = E.listed_positives(detector.graph, POSITIVE)
print("\nPositive diagnoses:", positives["diagnoses"])
print("Positive rules:     ", positives["rules"] or "(none — diagnoses only)")
display(inv[inv["type"].isin(["Diagnosis", "Rule"])].reset_index(drop=True))

## Processed STAFF III recordings

In [ ]:
index = E.list_processed_records(PROC_DIR)
display(index)
record_ids = [str(x) for x in index["record_id"].tolist()]
print(f"{len(record_ids)} records")

## A. Whole-recording rules vs balloon-up

Ground truth intervals come from `.event` inflation/deflation marks (`inflation_segments_from_events`), **not** `labels.segments`.

`evaluate_rules` treats an array symptom as true if **any** beat is true. Temporal scoring uses **trace beat ids / sample indexes**.

In [ ]:
strategy = get_active_strategy_name()
summaries = []
beat_frames = []
ste_rows = []

for rid in record_ids:
    signal, meta, labels = E.load_record(rid, PROC_DIR)
    fs = int(meta["fs"])
    channels = list(meta["channels"])
    payload = signal_from_arrays(signal, fs, channels)
    if WINDOW_S:
        i0 = int(WINDOW_START_S * fs)
        i1 = i0 + int(WINDOW_S * fs)
        payload = slice_signal(payload, i0, i1)

    cache_file = E.cache_path(CACHE_DIR, rid, strategy)
    cached = E.load_eval_cache(cache_file) if CACHE and not OVERWRITE_CACHE else None
    if cached is None:
        print(f"delineate+rules {rid}  n={payload['channels'][channels[0]].shape[0]} ...", flush=True)
        out = evaluate_recording(
            payload,
            detector=detector,
            include_trace=True,
        )
        if CACHE:
            E.save_eval_cache(
                cache_file,
                {"beats": out["beats"], "ir": out["ir"], "rules": out["rules"], "cleaned": out["cleaned"]},
            )
        ir, rules, cleaned, beats = out["ir"], out["rules"], out["cleaned"], out["beats"]
    else:
        print(f"cache {rid}", flush=True)
        ir, rules, cleaned, beats = cached["ir"], cached["rules"], cached["cleaned"], cached["beats"]

    scored = E.score_recording(record_id=rid, labels=labels, ir=ir, rules=rules, spec=POSITIVE)
    summaries.append({k: v for k, v in scored.items() if k not in {"detections", "beat_table", "latency_s"}})
    summaries[-1]["latency_s"] = scored["latency_s"]
    beat_frames.append(scored["beat_table"])

    artery = scored["occluded_artery"]
    leads = list(E.ARTERY_LEADS.get(artery, ("II",)))
    requested = [(lead, "ste60") for lead in leads if lead in cleaned["channels"]]
    if requested:
        metrics = compute_ecg_metrics(cleaned, beats, requested)
        for (lead, name), payload_m in metrics.items():
            values = payload_m.get("values")
            beats_idx = payload_m.get("heartbeat_indexes") or []
            if values is None:
                continue
            for beat_id, value in zip(beats_idx, values):
                beat = ir.get_beat(int(beat_id))
                sample = E.beat_sample(beat) if beat is not None else None
                ste_rows.append(
                    {
                        "record_id": rid,
                        "lead": lead,
                        "ste60": float(value),
                        "sample": sample,
                        "t_s": None if sample is None else sample / float(fs),
                        "occluded_artery": artery,
                        "phase": labels.get("phase"),
                    }
                )

summary = pd.DataFrame(summaries)
beats_all = pd.concat(beat_frames, ignore_index=True) if beat_frames else pd.DataFrame()
ste = pd.DataFrame(ste_rows)
display(summary)
print("beat-level", E.confusion_metrics(summaries))
print("record-level detections:", int(summary["pred_record"].sum()), "/", len(summary))

### Beat-level metrics (and by artery / phase)

In [ ]:
overall = E.confusion_metrics(summaries)
display(pd.DataFrame([overall]))

by_phase = []
for phase, grp in summary.groupby("phase"):
    row = E.confusion_metrics(grp.to_dict("records"))
    row["phase"] = phase
    by_phase.append(row)
display(pd.DataFrame(by_phase))

by_artery = []
for artery, grp in summary.dropna(subset=["occluded_artery"]).groupby("occluded_artery"):
    row = E.confusion_metrics(grp.to_dict("records"))
    row["occluded_artery"] = artery
    by_artery.append(row)
display(pd.DataFrame(by_artery))

## B. Territory vs `occluded_artery`

A territory hit: at least one triggering symptom lead is in the artery's lead set, **or** a resulting diagnosis maps to that artery.

| Artery | Leads |
|---|---|
| LAD | V1–V4, I, aVL |
| RCA | II, III, aVF |
| LCX | I, aVL, V5–V6 |

In [ ]:
terr = summary.copy()
terr["territory_rate"] = terr.apply(
    lambda r: (r["territory_hit"] / r["territory_n"]) if r["territory_n"] else float("nan"),
    axis=1,
)
display(terr[["record_id", "phase", "occluded_artery", "pred_record", "territory_hit", "territory_n", "latency_s"]])
print(
    "territory hits / inflation files with a detection:",
    int(terr["territory_hit"].sum()),
    "/",
    int(terr["territory_n"].sum()),
)

## C. STE60 in territory leads vs time

In [ ]:
import plotly.express as px

if ste.empty:
    print("No STE60 rows (delineation produced no complete P/QRS fiducials).")
else:
    display(ste.groupby(["record_id", "lead"])["ste60"].agg(["count", "mean", "median"]).reset_index())
    fig = px.scatter(
        ste,
        x="t_s",
        y="ste60",
        color="lead",
        facet_row="record_id",
        opacity=0.45,
        title="STE60 (mV) in occluded-artery leads",
        labels={"t_s": "Time (s)", "ste60": "STE60 (mV)"},
        height=180 * max(ste["record_id"].nunique(), 1) + 80,
    )
    fig.update_yaxes(matches=None)
    fig.show()

## Detection timeline vs balloon-up

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n = max(len(record_ids), 1)
fig = make_subplots(rows=n, cols=1, shared_xaxes=False, subplot_titles=record_ids, vertical_spacing=0.04)
for i, rid in enumerate(record_ids, start=1):
    _sig, meta, labels = E.load_record(rid, PROC_DIR)
    fs = int(meta["fs"])
    duration = int(labels["n_samples"]) / fs
    intervals = E.balloon_up_intervals(labels)
    fig.add_trace(
        go.Scatter(x=[0, duration], y=[0, 0], mode="lines", line=dict(color="#cccccc", width=8), showlegend=False),
        row=i, col=1,
    )
    for a, b in intervals:
        fig.add_vrect(x0=a / fs, x1=b / fs, fillcolor="#fc9272", opacity=0.35, line_width=0, row=i, col=1)
    hits = beats_all[(beats_all["record_id"] == rid) & beats_all["pred_positive"]]
    if len(hits):
        fig.add_trace(
            go.Scatter(
                x=hits["t_s"],
                y=[0.15] * len(hits),
                mode="markers",
                marker=dict(color="#08519c", size=6),
                name="detection",
                showlegend=(i == 1),
            ),
            row=i, col=1,
        )
    fig.update_yaxes(showticklabels=False, range=[-0.2, 0.4], row=i, col=1)
    fig.update_xaxes(title_text="Time (s)" if i == n else None, row=i, col=1)
fig.update_layout(height=90 * n + 80, title="Balloon-up (red) vs beat detections (blue)")
fig.show()